# Variational Autoencoders on MNIST

Companion notebook for the [Autoencoders and VAEs](https://codemia.ai/courses/deep_learning_foundations/autoencoders_and_vaes) lesson on Codemia.

**What you will build**: a convolutional VAE that learns a 16-dimensional latent space over MNIST digits, then samples brand-new digits from that latent space, interpolates between two real digits, and shows what *breaks* when you remove the reparameterization trick or the KL term.

**Time**: ~60 minutes  
**GPU required**: recommended (free-tier Colab T4 finishes the main training run in ~5 minutes; CPU works but is ~10x slower).

## Setup

In [ ]:
%pip install -q torch torchvision matplotlib

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {device}')
if device == 'cuda':
    print('  GPU:', torch.cuda.get_device_name(0))

## Concept recap

A VAE replaces the deterministic encoder of a vanilla autoencoder with a *distribution* over the latent space. For each input $x$ the encoder outputs $\mu(x)$ and $\log\sigma^2(x)$ — the parameters of a Gaussian $q(z\mid x)$ from which we sample $z$ before decoding. Training maximizes the evidence lower bound (ELBO):

$$
\mathcal{L} = \underbrace{\mathbb{E}_{q(z\mid x)}[\log p(x\mid z)]}_{\text{reconstruction}} \; - \; \underbrace{\mathrm{KL}\bigl(q(z\mid x)\,\Vert\,\mathcal{N}(0, I)\bigr)}_{\text{regularizer}}.
$$

The KL term is what makes the latent space *generative*. It pulls every per-input posterior towards the same standard-normal prior, so the prior actually covers the region the decoder can decode. Without it you get a regular autoencoder — recon is great, but sampling from $\mathcal{N}(0, I)$ at decode time gives garbage.

The reparameterization trick is the engineering bit: we cannot backprop through `z ~ N(mu, sigma)` because sampling is non-differentiable. We rewrite it as `z = mu + sigma * eps` with `eps ~ N(0, 1)` — now `mu` and `sigma` are deterministic functions of the input, and the only stochasticity is in `eps`, which we treat as exogenous noise. Gradients flow cleanly through `mu` and `sigma`.

## Implementation

### Step 1 — Data

MNIST: 60k training images of 28x28 grayscale digits. We binarize-via-Bernoulli at decode time but keep inputs as floats in [0,1] for stable convs.

In [ ]:
transform = torchvision.transforms.ToTensor()
train_ds = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
test_ds = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)

# Batch 128 keeps the KL term well-estimated without blowing free-tier memory.
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

x_sample, y_sample = next(iter(train_loader))
print('one batch:', x_sample.shape, 'labels:', y_sample.shape, 'pixel range:', x_sample.min().item(), x_sample.max().item())

### Step 2 — Convolutional encoder

Three stride-2 convs take 28x28 → 14x14 → 7x7 → 4x4 (rounding up via padding). After flatten, two linear heads produce `mu` and `logvar`. We predict `logvar` rather than `sigma` so the model can output any positive variance via `exp` without learning a constraint.

In [ ]:
class ConvEncoder(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        # 1x28x28 -> 32x14x14 -> 64x7x7 -> 128x4x4
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(inplace=True),
        )
        self.flatten = nn.Flatten()
        # Two heads: shared trunk learns features, then we split into the two parameters
        # of the per-input Gaussian. logvar (not sigma) because it is unconstrained in R.
        self.fc_mu = nn.Linear(128 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(128 * 4 * 4, latent_dim)
    def forward(self, x):
        h = self.flatten(self.conv(x))
        return self.fc_mu(h), self.fc_logvar(h)

### Step 3 — Convolutional decoder

Mirror image: a linear projection back to 128x4x4, then three transposed convs back to 1x28x28. Final `sigmoid` keeps outputs in [0,1] so we can use BCE recon.

In [ ]:
class ConvDecoder(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 128 * 4 * 4)
        # output_padding nudges spatial dims back to 7, 14, 28 exactly.
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=0), nn.ReLU(inplace=True),  # 4 -> 7
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), nn.ReLU(inplace=True),   # 7 -> 14
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1),                           # 14 -> 28
        )
    def forward(self, z):
        h = self.fc(z).view(-1, 128, 4, 4)
        # Sigmoid because BCE expects probabilities. Logits would also work with BCEWithLogits.
        return torch.sigmoid(self.deconv(h))

### Step 4 — VAE module + the reparameterization trick

`stochastic=False` lets us run the variation that disables sampling. With `stochastic=True` we use `z = mu + sigma * eps` so gradients flow through `mu` and `sigma`. With `stochastic=False` we just pass `mu` straight through — this is what most readers think a VAE "is" until they realize it collapses to a pure autoencoder.

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=16, stochastic=True):
        super().__init__()
        self.latent_dim = latent_dim
        self.stochastic = stochastic
        self.encoder = ConvEncoder(latent_dim)
        self.decoder = ConvDecoder(latent_dim)
    def reparameterize(self, mu, logvar):
        # The trick: pull the randomness *out* of the sampling op so gradients pass through mu, logvar.
        # std = exp(0.5 * logvar) because logvar = log(sigma^2) -> sigma = exp(logvar/2).
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps
    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar) if self.stochastic else mu
        x_recon = self.decoder(z)
        return x_recon, mu, logvar

### Step 5 — ELBO loss

Two pieces:

- **Reconstruction**: per-pixel BCE summed over the image, averaged over the batch. Summing (not averaging) over pixels keeps the recon term on the same scale as the KL — averaging would shrink it ~784x and let the KL dominate.
- **KL to N(0, I)**: the closed-form expression `-0.5 * sum(1 + logvar - mu^2 - exp(logvar))`. This pulls every per-input posterior toward the standard-normal prior. Without it, the encoder is free to put each digit on its own remote planet — recon stays great but the prior is uncovered.

In [ ]:
def vae_loss(x, x_recon, mu, logvar, kl_weight=1.0):
    # BCE summed over pixels, averaged over batch. F.binary_cross_entropy with reduction='sum'
    # and divide by batch size: each image contributes its full per-pixel sum.
    bsz = x.size(0)
    recon = F.binary_cross_entropy(x_recon, x, reduction='sum') / bsz
    # Closed-form KL[q(z|x) || N(0, I)] for a diagonal Gaussian q.
    # Sum over latent dims, average over batch -> nats per image.
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / bsz
    total = recon + kl_weight * kl
    return total, recon, kl

Sanity check: one forward pass, print component shapes and values so we know the wiring is right before we burn 5 epochs of compute.

In [ ]:
vae_check = VAE(latent_dim=16).to(device)
x_check = x_sample.to(device)
x_recon, mu, logvar = vae_check(x_check)
print('input :', x_check.shape)
print('recon :', x_recon.shape, ' should match input')
print('mu    :', mu.shape, ' (batch, latent_dim)')
print('logvar:', logvar.shape)
loss, recon, kl = vae_loss(x_check, x_recon, mu, logvar)
print(f'untrained: loss={loss.item():.2f}  recon={recon.item():.2f}  kl={kl.item():.4f}')
print(f'param count: {sum(p.numel() for p in vae_check.parameters()):,}')

## Experiment: train the main VAE

5 epochs, Adam, lr=1e-3. On a T4 this finishes in ~5 minutes. We track recon and KL separately — watching them evolve is more informative than the combined loss.

In [ ]:
def train_vae(model, epochs=5, lr=1e-3, kl_weight=1.0, log_every=100):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'loss': [], 'recon': [], 'kl': []}
    model.train()
    for epoch in range(epochs):
        for step, (x, _) in enumerate(train_loader):
            x = x.to(device, non_blocking=True)
            x_recon, mu, logvar = model(x)
            loss, recon, kl = vae_loss(x, x_recon, mu, logvar, kl_weight=kl_weight)
            opt.zero_grad()
            loss.backward()
            opt.step()
            history['loss'].append(loss.item())
            history['recon'].append(recon.item())
            history['kl'].append(kl.item())
            if step % log_every == 0:
                print(f'  epoch {epoch} step {step:4d}  loss={loss.item():.2f}  recon={recon.item():.2f}  kl={kl.item():.3f}')
    return history

In [ ]:
torch.manual_seed(0)
vae_main = VAE(latent_dim=16, stochastic=True).to(device)
hist_main = train_vae(vae_main, epochs=5, kl_weight=1.0)

In [ ]:
# Plot the two loss components separately. KL should rise quickly and then stabilize as the
# encoder spreads the posteriors out; recon should fall throughout.
fig, ax = plt.subplots(1, 3, figsize=(12, 3.2))
ax[0].plot(hist_main['loss']); ax[0].set_title('total loss'); ax[0].grid(alpha=0.3)
ax[1].plot(hist_main['recon'], color='tab:orange'); ax[1].set_title('reconstruction (BCE/img)'); ax[1].grid(alpha=0.3)
ax[2].plot(hist_main['kl'], color='tab:green'); ax[2].set_title('KL to N(0, I)'); ax[2].grid(alpha=0.3)
for a in ax: a.set_xlabel('step')
plt.tight_layout(); plt.show()

## Visualizations

### (a) Reconstructions

First row: real digits from the test set. Second row: VAE reconstructions. They should be recognizable but slightly blurry — that blurriness is the *price* of the prior regularization, and is inherent to a Gaussian-likelihood VAE.

In [ ]:
@torch.no_grad()
def show_reconstructions(model, n=10):
    model.eval()
    x_test, _ = next(iter(test_loader))
    x_test = x_test[:n].to(device)
    x_recon, _, _ = model(x_test)
    fig, ax = plt.subplots(2, n, figsize=(n * 1.0, 2.2))
    for i in range(n):
        ax[0, i].imshow(x_test[i, 0].cpu(), cmap='gray'); ax[0, i].axis('off')
        ax[1, i].imshow(x_recon[i, 0].cpu(), cmap='gray'); ax[1, i].axis('off')
    ax[0, 0].set_ylabel('real', rotation=0, labelpad=20); ax[1, 0].set_ylabel('recon', rotation=0, labelpad=24)
    plt.suptitle('Reconstructions: top row real, bottom row VAE')
    plt.tight_layout(); plt.show()

show_reconstructions(vae_main)

### (b) Prior samples

Sample 64 vectors from $\mathcal{N}(0, I)$ and decode. These should look like recognizable digits — that is the entire point of the KL regularizer.

In [ ]:
@torch.no_grad()
def show_prior_samples(model, n=64, title='Samples from N(0, I) prior'):
    model.eval()
    z = torch.randn(n, model.latent_dim, device=device)
    x_gen = model.decoder(z).cpu()
    grid = torchvision.utils.make_grid(x_gen, nrow=int(math.sqrt(n)), pad_value=1.0)
    plt.figure(figsize=(6, 6))
    plt.imshow(grid.permute(1, 2, 0).numpy()); plt.axis('off'); plt.title(title); plt.show()

show_prior_samples(vae_main)

### (c) Latent-space interpolation

Pick two real digits, encode each to its $\mu$, walk a straight line between the two $\mu$'s in 10 steps, and decode each step. A good VAE produces a smooth morph between digit classes — the latent space is *continuous* in a way that pixel space is not.

In [ ]:
@torch.no_grad()
def show_interpolation(model, n_steps=10):
    model.eval()
    x_test, y_test = next(iter(test_loader))
    # Pick two clearly different classes for a more dramatic morph.
    i_a = (y_test == 1).nonzero(as_tuple=True)[0][0].item()
    i_b = (y_test == 7).nonzero(as_tuple=True)[0][0].item()
    x_a = x_test[i_a:i_a+1].to(device); x_b = x_test[i_b:i_b+1].to(device)
    mu_a, _ = model.encoder(x_a); mu_b, _ = model.encoder(x_b)
    alphas = torch.linspace(0, 1, n_steps, device=device).unsqueeze(1)
    # Linear interp in latent space — only sensible because the KL keeps the latent space packed.
    zs = (1 - alphas) * mu_a + alphas * mu_b
    imgs = model.decoder(zs).cpu()
    fig, ax = plt.subplots(1, n_steps, figsize=(n_steps * 1.0, 1.2))
    for i in range(n_steps):
        ax[i].imshow(imgs[i, 0], cmap='gray'); ax[i].axis('off')
    plt.suptitle('Linear interpolation in latent space (1 -> 7)')
    plt.tight_layout(); plt.show()

show_interpolation(vae_main)

### (d) PCA scatter of the latent space

Encode the entire test set, then project the 16-d $\mu$'s down to 2D with PCA. Color each point by its digit class. We expect to see distinct-but-overlapping clusters — the prior pressure keeps everything packed near the origin, but the encoder still has incentive to separate classes for the recon term.

In [ ]:
@torch.no_grad()
def latent_pca_scatter(model, max_n=5000):
    model.eval()
    mus, ys = [], []
    for x, y in test_loader:
        x = x.to(device)
        mu, _ = model.encoder(x)
        mus.append(mu.cpu()); ys.append(y)
        if sum(m.size(0) for m in mus) >= max_n:
            break
    mus = torch.cat(mus)[:max_n]
    ys = torch.cat(ys)[:max_n]
    # PCA via SVD: center, then project onto top-2 right singular vectors.
    mus_c = mus - mus.mean(0, keepdim=True)
    _, _, V = torch.linalg.svd(mus_c, full_matrices=False)
    coords = mus_c @ V[:2].T
    plt.figure(figsize=(7, 6))
    sc = plt.scatter(coords[:, 0], coords[:, 1], c=ys.numpy(), cmap='tab10', s=6, alpha=0.7)
    plt.colorbar(sc, ticks=range(10), label='digit class')
    plt.title(f'PCA of {max_n} test latents (16-d mu -> 2-d)')
    plt.xlabel('PC1'); plt.ylabel('PC2'); plt.grid(alpha=0.3); plt.show()

latent_pca_scatter(vae_main)

## Variations

### Variation 1 — Disable the reparameterization trick

We rebuild the model with `stochastic=False`. Now the forward pass uses `mu` directly with no sampling. Gradients flow as cleanly as ever — but the encoder no longer needs to output a *distribution* that survives Gaussian noise, only a *point*. That removes the implicit "smear" that forces nearby latents to decode similarly. Result: the per-input posteriors collapse to deterministic points, large gaps open up in the latent space, and prior samples fall into those gaps.

In [ ]:
torch.manual_seed(0)
vae_det = VAE(latent_dim=16, stochastic=False).to(device)
hist_det = train_vae(vae_det, epochs=5, kl_weight=1.0, log_every=200)

In [ ]:
# Recon should still look fine — the model has effectively become a regularized autoencoder.
show_reconstructions(vae_det)
# Prior samples should be noticeably worse: many samples decode into ambiguous blurry blobs.
show_prior_samples(vae_det, title='Prior samples — stochastic=False (no reparam trick)')

### Variation 2 — Drop the KL term entirely

Set `kl_weight=0.0`. The model is now a pure autoencoder. Recon will be *better* than the VAE's. But there is no pressure on the latent space's geometry, so $\mathcal{N}(0, I)$ no longer matches what the decoder expects, and prior samples become noise. This is the cleanest demonstration of why the KL term is the *generative* part of a VAE.

In [ ]:
torch.manual_seed(0)
ae_pure = VAE(latent_dim=16, stochastic=True).to(device)
hist_ae = train_vae(ae_pure, epochs=5, kl_weight=0.0, log_every=200)

In [ ]:
# Recon: probably the sharpest of the three models — no KL pressure to compromise.
show_reconstructions(ae_pure)
# Prior samples: garbage. The encoder put each digit wherever it liked; N(0, I) covers none of it.
show_prior_samples(ae_pure, title='Prior samples — kl_weight=0.0 (pure autoencoder)')

Side-by-side comparison of the three models' final recon and KL on the same training data, just to make the trade-off concrete.

In [ ]:
def final_means(history, last_n=200):
    # Average the last few hundred steps to smooth out per-batch noise.
    import statistics as st
    return st.mean(history['recon'][-last_n:]), st.mean(history['kl'][-last_n:])

for name, hist in [('VAE (stochastic, kl=1)', hist_main),
                   ('VAE (stochastic=False, kl=1)', hist_det),
                   ('AE (stochastic, kl=0)', hist_ae)]:
    r, k = final_means(hist)
    print(f'{name:35s}  recon={r:6.2f}  kl={k:6.3f}')

## What to try next

- **β-VAE**: scale the KL term by `β > 1` (encourages disentangled axes — each latent dim captures a single factor of variation) or `β < 1` (sharper recon, less coherent prior). Re-run the PCA scatter and watch the cluster geometry change.
- **VQ-VAE**: replace the continuous latent with a learned discrete codebook + straight-through estimator. Same skeleton, different bottleneck — and it scales to images that a Gaussian VAE smears.
- **Conditional VAE**: feed the digit label into both encoder and decoder. Now you can sample digits of a *specific* class. Compare class-conditional samples to the unconditional ones above.
- **Likelihood upgrade**: swap BCE for a discretized logistic mixture (PixelCNN++ style). The blurriness in your reconstructions is largely a Gaussian-likelihood artifact, not a latent-space failure.